In [1]:
%pip install numpy


[notice] A new release of pip is available: 26.1 -> 26.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np

matrix_game = np.array([
    [ 1.0, -1.0],
    [-1.0,  1.0]
])

def get_payoffs(a0, a1):
    reward_p0 = matrix_game[a0, a1]
    reward_p1 = -reward_p0
    return reward_p0, reward_p1


r0, r1 = get_payoffs(0, 0)
print(f"Joueur 0 : {r0}, Joueur 1 : {r1}") 

class MatrixGame(ParallelEnv):
    metadata = {'render_modes': ['human']}

    def get_payoffs(self, a0, a1):
        reward_p0 = self.reward_matrix[a0, a1]
        reward_p1 = -reward_p0
        return reward_p0, reward_p1


    def __init__(self, reward_matrix):
        self.agents = ["player_0", "player_1"]
        self.possible_agents = self.agents[:]
        self.reward_matrix = reward_matrix

        # Action space is Discrete(N) where N is the size of the matrix (e.g., 2x2 -> N=2)
        self.action_spaces = {agent: spaces.Discrete(reward_matrix.shape[0]) for agent in self.agents}
        # Observation space is Discrete(1) because the game is stateless (only one observation/state: 0)
        self.observation_spaces = {agent: spaces.Discrete(1) for agent in self.agents}

    def reset(self, seed=None, options=None):
        self.agents = self.possible_agents[:]
        self.rewards = {agent: 0.0 for agent in self.agents}
        self.dones = {agent: False for agent in self.agents}
        self.infos = {agent: {} for agent in self.agents}
        self.state = None  # Stateless environment

        return {agent: 0 for agent in self.agents}  # Observations are dummy since the environment is stateless

    def step(self, actions):
        if not self.agents:
            # Handle the case where the episode is over
            return {}, {}, {}, {}, {}

        # Extract actions (player_0's action is row index, player_1's action is column index)
        action_0 = actions[self.agents[0]]
        action_1 = actions[self.agents[1]]

        # Get rewards from the reward matrix: reward_matrix[row][col] -> (R0, R1)
        reward_0, reward_1 = self.get_payoffs(action_0, action_1)

        # Update rewards
        self.rewards = {self.agents[0]: reward_0, self.agents[1]: reward_1}

        # Since this is a one-shot game, mark agents as done
        self.dones = {agent: True for agent in self.agents}
        self.truncateds = {agent: False for agent in self.agents}

        # No additional info in this case
        self.infos = {agent: {} for agent in self.agents}

        # Observations are dummy since the environment is stateless
        observations = {agent: 0 for agent in self.agents}

        return observations, self.rewards, self.dones, self.truncateds, self.infos

    def render(self):
        pass

    def close(self):
        pass

Joueur 0 : 1.0, Joueur 1 : -1.0
